# BoolQ replication with four fixed students


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, time, json, hashlib, zipfile, importlib.util, subprocess
from datetime import datetime
from zoneinfo import ZoneInfo
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import numpy as np

SOURCE = Path('/content/drive/MyDrive/teacher_risk_real_tasks/782e9756c1c8b35c/env_f6f4a53d6e79')
ROOT = SOURCE/'real_teacher_bias_intervention_v1_c5a072a00b7f/env_f6f4a53d6e79'
RIDGE = ROOT/'llm_features_ridge_2teachers_v1'
PAIRS = ROOT/'real_teacher_pairs_ridge_v2_fixed'
OUT = ROOT/'boolq_fresh_olmo_qwen_m4_v1'
if not (RIDGE/'features/train/complete.npz').is_file():
    raise FileNotFoundError(f'Mount the original Drive. Required training cache: {RIDGE}/features/train/complete.npz')
OUT.mkdir(exist_ok=True)
TZ = ZoneInfo('Europe/Amsterdam')
start_time = datetime.now(TZ)
pause_at = start_time.replace(hour=0, minute=58, second=0, microsecond=0) if start_time.hour == 0 else None
def time_check():
    if pause_at and datetime.now(TZ) >= pause_at:
        raise RuntimeError('Time budget reached. Completed blocks are on Drive. Resume after 01:00.')
def read(path):
    return json.loads(path.read_text())
def obj_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()
def write_json(path, obj):
    temp = path.with_suffix('.writing')
    temp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    os.replace(temp, path)
def freeze(path, obj):
    if path.exists():
        if read(path) != obj:
            raise ValueError(f'Existing record differs: {path}')
    else:
        write_json(path, obj)
def write_npz(path, **arrays):
    temp = path.with_suffix('.writing')
    with temp.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temp, path)
def load_array(path, key):
    with np.load(path, allow_pickle=False) as archive:
        return archive[key].copy()
def required_from_pairs(name):
    p = PAIRS/name
    if p.exists():
        return p
    with zipfile.ZipFile(ROOT/(PAIRS.name+'_results.zip')) as archive:
        member = next(n for n in archive.namelist() if n.endswith('/'+name))
        p = Path('/content')/('fresh_m4_'+Path(name).name)
        p.write_bytes(archive.read(member))
        return p
QWEN = 'Qwen/Qwen2.5-3B-Instruct'
OLMO = 'allenai/OLMo-2-0425-1B-Instruct'
FEATURE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REV = '7ae557604adf67be50417f59c2c2f167def9a775'
DATA_REV = '35b264d03638db9f4ce671b711558bf7ff0f80d5'
sel = read(required_from_pairs('selection_frozen.json'))
choice, = [r for r in sel['choices'] if r['M']==4 and r['pair']==[QWEN,OLMO]]
private = np.array(choice['private']); mixed = np.array(choice['population'])
assert private.tolist()==[1,1,1,1] and mixed.tolist()==[1,0,1,1]
design = dict(teachers=[QWEN,OLMO], private=private.tolist(), mixed=mixed.tolist(),
              primary='private minus mixed raw aggregate MSE', source_split='BoolQ/train',
              target_rows=1600, sample_rule='first 1600 eligible rows in a seeded permutation, or all eligible if fewer',
              row_seed=17202611, model_revision=MODEL_REV, dataset_revision=DATA_REV,
              inference='paired bootstrap by original passage, conditional on the four student slots',
              secondary=['all wrong','majority accuracy (half credit on ties)','uniform-routing accuracy','uniform-routing MSE'],
              preference='individual Qwen MSE minus individual OLMo MSE for each slot')
freeze(OUT/'design.json',design)
print('Assignment verified. Drive output:', OUT)


In [2]:
protocol=read(RIDGE/'protocol.json')
assert protocol['revision']==MODEL_REV
train=read(ROOT/'data/train.json')
assert obj_hash(train)==protocol['source_data_hashes']['train']
ytrain=np.array([r['y'] for r in train])
pca=RIDGE/'pca.npz'
mu=load_array(pca,'mu'); projection=load_array(pca,'projection')
def transform(x):
    return (x.astype(np.float64)-mu) @ projection
z=transform(load_array(RIDGE/'features/train/complete.npz','x'))
targets=np.column_stack([load_array(ROOT/'teachers/original_targets.npz','train')[0],
                         load_array(required_from_pairs('teachers/OLMo-2-0425-1B-Instruct.npz'),'p')])
assert targets.shape==(2048,2) and z.shape==(2048,128)
weights=[]; offsets=[]
for slot in range(4):
    ids=np.sort(np.random.default_rng(2026111805+slot).choice(len(z),1536,replace=False))
    x=z[ids]; t=(0.3*ytrain[:,None]+0.7*targets)[ids]
    mx=x.mean(0); mt=t.mean(0); xc=x-mx
    w=np.linalg.solve(xc.T@xc/len(x)+0.1*np.eye(x.shape[1]), xc.T@(t-mt)/len(x))
    weights.append(w); offsets.append(mt-mx@w)
weights=np.stack(weights); offsets=np.stack(offsets)
def predict_heads(features):
    return transform(features) @ weights + offsets[:,None,:] # slot, item, teacher
def metrics(scores, labels):
    error=scores-labels
    wrong=(scores>=0.5)!=labels.astype(bool)
    votes=(scores>=0.5).sum(axis=0)
    vote_acc=np.where(votes==2,0.5,(votes>2)==labels.astype(bool))
    return np.column_stack([error.mean(0)**2, wrong.all(0), vote_acc, 1-wrong.mean(0), (error**2).mean(0)])
def policies(bank, labels):
    return [metrics(np.stack([bank[i,:,j] for i,j in enumerate(a)]),labels) for a in (private,mixed)]
oldrows=read(ROOT/'controlled_teacher_choice_2x4_v2/data/test_new.json')
assert obj_hash(oldrows)==protocol['source_data_hashes']['test']
oldfeatures=load_array(RIDGE/'features/test/complete.npz','x')
oldbank=predict_heads(oldfeatures)
oldpriv,oldmix=policies(oldbank,np.array([r['y'] for r in oldrows]))
import csv
with required_from_pairs('TEST_ALL_PAIRS.csv').open() as f:
    ref,=[r for r in csv.DictReader(f) if r['preferred']==QWEN.split('/')[-1] and r['alternative']==OLMO.split('/')[-1] and r['M']=='4']
np.testing.assert_allclose((oldpriv-oldmix).mean(0)[:3],
    [float(ref['aggregate_mse_gain']),float(ref['all_wrong_reduction']),-float(ref['vote_accuracy_gain'])],atol=1e-9,rtol=0)
print('Old comparison reproduced. Eight original heads ready.')


Old comparison reproduced. Eight original heads ready.


In [ ]:
for pkg in ('datasets','transformers','accelerate'):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',pkg if pkg!='transformers' else 'transformers==4.57.3'])
from transformers import AutoTokenizer
tok=AutoTokenizer.from_pretrained(FEATURE_MODEL,revision=MODEL_REV)
if tok.pad_token_id is None: tok.pad_token=tok.eos_token
tok.padding_side='right'
def norm(s):
    return ' '.join(s.lower().split())
def prefix(s):
    return norm(' '.join(s.split()[:180]))
def encode(context,question):
    msg='Read the text and answer the question. Answer only Yes or No.\n\nText:\n'+context+'\n\nQuestion: '+question+'\nAnswer:'
    prompt=tok.apply_chat_template([{'role':'user','content':msg}],tokenize=False,add_generation_prompt=True)
    return prompt,tok.encode(prompt,add_special_tokens=False)
manifest_path=OUT/'question_manifest.json'
sample_path=OUT/'questions.json'
if manifest_path.exists():
    manifest=read(manifest_path); rows=read(sample_path)
    assert obj_hash(rows)==manifest['rows_hash'] and manifest['design_hash']==obj_hash(design)
else:
    from datasets import load_dataset
    old_sources=[ROOT/'data/train.json',ROOT/'data/diagnostic.json', ROOT/'data/fresh_evaluation.json',
                 ROOT/'controlled_teacher_choice_2x4_v2/data/test_new.json', SOURCE/'final_suite_cells_v1/boolq/data/final.json',
                 SOURCE/'overnight_6be09513b41f/data/explored_selection.json']
    seen_q=set(); seen_p=set(); history={}
    for path in old_sources:
        records=read(path); history[str(path.relative_to(SOURCE))]=len(records)
        for r in records:
            seen_q.add(norm(r['question']))
            for key in ('context','original_context','passage'):
                if r.get(key): seen_p.add(prefix(r[key]))
    raw=load_dataset('google/boolq',revision=DATA_REV,split='train')
    for r in raw:
        if norm(r['question']) in seen_q: seen_p.add(prefix(r['passage']))
    enc=read(ROOT/'answer_encoding.json')
    answers=list(zip(enc['texts'][FEATURE_MODEL],enc['ids'][FEATURE_MODEL]))
    rows=[]; visited=set()
    for index in np.random.default_rng(design['row_seed']).permutation(len(raw)):
        r=raw[int(index)]; q=norm(r['question']); passage=r['passage']
        if q in seen_q or q in visited or prefix(passage) in seen_p: continue
        words=passage.split()[:180]
        while words:
            context=' '.join(words); prompt,ids=encode(context,r['question'])
            if len(ids)<=384: break
            words=words[:-max(1,len(words)//12)]
        if not words or any(tok.encode(prompt+t,add_special_tokens=False)!=ids+[i] for t,i in answers): continue
        visited.add(q)
        rows.append(dict(question=r['question'],context=context,y=int(r['answer']),source_id=int(index),
                         cluster=hashlib.sha256(norm(passage).encode()).hexdigest()))
        if len(rows)==design['target_rows']: break
    if not rows: raise RuntimeError('No eligible rows in the pinned dataset.')
    manifest=dict(rows_hash=obj_hash(rows),design_hash=obj_hash(design),questions=len(rows),
                  passages=len({r['cluster'] for r in rows}), prior_files=history)
    write_json(sample_path,rows); freeze(manifest_path,manifest)
print('Fixed fresh sample:',len(rows),'questions,',manifest['passages'],'passages.')


In [ ]:
import torch
from transformers import AutoModelForCausalLM
from importlib.metadata import version
blockdir=OUT/'blocks'; blockdir.mkdir(exist_ok=True)
signature=obj_hash(dict(rows=manifest['rows_hash'],design=design,
   model=MODEL_REV, numerical_mode='float32 eager', transformers=version('transformers'),torch=torch.__version__,
   weights=hashlib.sha256(weights.tobytes()+offsets.tobytes()+mu.tobytes()+projection.tobytes()).hexdigest()))
blocks=[]; pending=[]
for lo in range(0,len(rows),32):
    hi=min(lo+32,len(rows)); path=blockdir/f'{lo:05d}_{hi:05d}.npz'
    blocks.append(path)
    if not path.exists(): pending.append((lo,hi,path)); continue
    with np.load(path) as a:
        if str(a['signature'])!=signature or a['scores'].shape!=(4,hi-lo,2) or not np.isfinite(a['scores']).all():
            raise RuntimeError(f'Saved block is incompatible: {path}')
if pending:
    time_check()
    if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime in Colab, then run again.')
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    model=AutoModelForCausalLM.from_pretrained(FEATURE_MODEL,revision=MODEL_REV,torch_dtype=torch.float32,
          attn_implementation='eager',low_cpu_mem_usage=True).to('cuda').eval()
    # Check that the current runtime extracts the same features as the original run.
    with torch.inference_mode():
        ids=encode(oldrows[0]['context'],oldrows[0]['question'])[1]
        inp=torch.tensor([ids],device='cuda')
        h=model.model(input_ids=inp,attention_mask=torch.ones_like(inp),
                      position_ids=torch.arange(len(ids),device='cuda')[None,:],
                      use_cache=False,return_dict=True).last_hidden_state
        check=h[0,-1].float().cpu().numpy()
        delta=np.max(np.abs(predict_heads(check[None,:])-oldbank[:,:1,:]))
        if not np.isfinite(delta) or delta>2e-4:
            raise RuntimeError(f'Old prompt did not replay on this GPU (score difference {delta}). Check the original Transformers environment.')
    print('GPU replay passed. Maximum original-score discrepancy:',float(delta))
    bench=time.monotonic(); batch=4
    for k,(lo,hi,path) in enumerate(pending,1):
        time_check()
        seq=[encode(r['context'],r['question'])[1] for r in rows[lo:hi]]
        features=[]; j=0
        while j<len(seq):
            group=seq[j:j+batch]; n=len(group); width=max(map(len,group))
            tokens=torch.full((n,width),tok.pad_token_id,dtype=torch.long,device='cuda')
            mask=torch.zeros_like(tokens)
            for i,s in enumerate(group):
                tokens[i,:len(s)]=torch.tensor(s,device='cuda'); mask[i,:len(s)]=1
            def forward():
                with torch.inference_mode():
                    h=model.model(input_ids=tokens,attention_mask=mask,position_ids=(mask.cumsum(-1)-1).clamp_min(0),
                                  use_cache=False,return_dict=True).last_hidden_state
                    return h[torch.arange(n,device='cuda'),mask.sum(1)-1].float().cpu().numpy()
            try: f=forward()
            except torch.cuda.OutOfMemoryError:
                if batch==1: raise
                batch=max(batch//2,1); f=None
            if f is None:
                torch.cuda.empty_cache(); continue
            features.append(f); j+=n
        scores=predict_heads(np.concatenate(features))
        if not np.isfinite(scores).all(): raise FloatingPointError(f'Nonfinite scores at {lo}')
        write_npz(path,scores=scores,signature=np.array(signature))
        eta=(time.monotonic()-bench)*(len(pending)-k)/k/60
        print(f'{hi}/{len(rows)} saved to Drive. Estimated scoring time left: {eta:.1f} min.',flush=True)
    del model
    torch.cuda.empty_cache()
else: print('Reusing all completed prediction blocks.')


In [ ]:
bank=np.concatenate([load_array(p,'scores') for p in blocks],axis=1)
assert bank.shape==(4,len(rows),2)
y=np.array([r['y'] for r in rows])
a,b=policies(bank,y)
names=['aggregate MSE','all wrong rate','vote accuracy','uniform routing accuracy','uniform routing MSE']
sign=np.array([1,1,-1,-1,1])
own=((bank[:,:,0]-y)**2-(bank[:,:,1]-y)**2).T
diff=np.column_stack([(a-b)*sign,own])
def bootstrap(x,cluster):
    _,i=np.unique(cluster,return_inverse=True); n=int(i.max()+1); counts=np.bincount(i)
    sums=np.stack([np.bincount(i,weights=v) for v in x.T],1)
    rng=np.random.default_rng(2026111817); draws=[]
    for j in range(40):
        w=rng.multinomial(n,np.full(n,1/n),size=100)
        draws.append((w@sums)/(w@counts)[:,None])
    return np.quantile(np.vstack(draws),[.025,.975],axis=0)
lo,hi=bootstrap(diff,[r['cluster'] for r in rows])
output=[]
for j,name in enumerate(names):
    output.append(dict(metric=name,all_OLMo=float(a[:,j].mean()),three_OLMo_one_Qwen=float(b[:,j].mean()),
                       gain=float(diff[:,j].mean()),low95=float(lo[j]),high95=float(hi[j])))
margins=[dict(slot=j+1,Qwen_minus_OLMo_MSE=float(own[:,j].mean()),low95=float(lo[5+j]),high95=float(hi[5+j])) for j in range(4)]
results=dict(questions=len(rows),passages=manifest['passages'],metrics=output,individual_margins=margins,
             design_hash=obj_hash(design),sample_hash=manifest['rows_hash'],prediction_signature=signature)
write_json(OUT/'results.json',results)
import pandas as pd
table=pd.DataFrame(output); table.to_csv(OUT/'results.csv',index=False)
archive=OUT/'fresh_m4_results.zip'
temp=OUT/'fresh_m4_results.writing'
with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED) as zfile:
    for p in [OUT/'design.json',manifest_path,sample_path,OUT/'results.json',OUT/'results.csv',*blocks]:
        zfile.write(p,p.relative_to(OUT))
os.replace(temp,archive)
display(table.round(6),pd.DataFrame(margins).round(6))
print('Saved:',archive)
